<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## Cadenas de Markov de Tiempo Continuo: Modelamiento
### T2.2 · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Contenido
1. Metodología de modelamiento (5 pasos)
2. **Escenario 1:** Cola M/M/1 — restaurante (una variable, tasas constantes)
3. **Escenario 2:** Cola M/M/1/K — clínica dental (capacidad finita)
4. **Escenario 3:** Cola M/M/c — call center (múltiples servidores)
5. **Escenario 4:** Reparación de máquinas (dos variables → reducción)
6. **Escenario 5:** Red de colas con dos estaciones (modelo implícito)
7. Explícito vs. implícito
8. Ejercicio propuesto

## Metodología de modelamiento
<div class='defn'>
<strong>Meta:</strong> Dada una descripción textual de un sistema estocástico, construir sistemáticamente su CMTC.
</div>

**Proceso en 5 pasos:**

| Paso | Acción |
|------|--------|
| 1 | Identificar las **variables aleatorias** relevantes |
| 2 | Definir el **espacio de estados** de cada variable |
| 3 | Construir la **variable de estado conjunta** y su espacio |
| 4 | Identificar las **transiciones posibles** y sus tasas |
| 5 | Ensamblar la **matriz generadora Q** |

<div class='nota'>
Avanzaremos de escenarios simples (una variable) a complejos (dos variables con acoplamiento), y de matrices <strong>explícitas</strong> a representaciones <strong>implícitas</strong> programables.
</div>

## Escenario 1 — Cola M/M/1: restaurante
<div class='defn'>
Un pequeño restaurante tiene <strong>un solo cajero</strong>. Los clientes llegan con tasa λ = 3 clientes/min. El cajero atiende a tasa μ = 5 clientes/min. La fila no tiene límite de capacidad.
</div>

**Aplicando los 5 pasos:**

- **Variable aleatoria:** N(t) = número de clientes en el sistema
- **Espacio de estados:** S = {0, 1, 2, 3, …} = ℕ₀
- **Eventos:** llegada (n → n+1, tasa λ); servicio (n → n−1, tasa μ, solo si n ≥ 1)
- **Forma implícita de Q:**

$$q_{n,n+1} = \lambda, \quad q_{n,n-1} = \mu \cdot \mathbb{1}(n \geq 1), \quad q_{nn} = -\lambda - \mu \cdot \mathbb{1}(n \geq 1)$$

<div class='nota'>
Es un proceso de <strong>nacimiento y muerte</strong>: solo hay transiciones entre estados adyacentes. Las tasas son constantes e independientes del estado.
</div>

In [1]:
import numpy as np
from jmarkov.ctmc import ctmc

lam, mu = 3.0, 5.0
N = 10   # truncar para visualizar

# Construcción implícita de Q
Q1 = np.zeros((N+1, N+1))
for n in range(N+1):
    if n < N:
        Q1[n, n+1] = lam
    if n > 0:
        Q1[n, n-1] = mu
np.fill_diagonal(Q1, -Q1.sum(axis=1))

print("Matriz Q (M/M/1, primeras 5 filas):")
print(np.round(Q1[:5, :5], 1))
print(f"\nForma implícita: q_{{n,n+1}}={lam}, q_{{n,n-1}}={mu}  (tasa diagonal: q_00={Q1[0,0]}, q_11={Q1[1,1]})")

mc1 = ctmc(Q1, np.arange(N+1))
print(f"\njmarkov — ¿Es ergódica? {mc1.is_ergodic()}")
pi1 = mc1.steady_state()
print(f"Distribución estacionaria π∞ (estados 0..5): {pi1[:6].round(4)}")
rho = lam/mu
print(f"Verificación: ρ = λ/μ = {rho:.2f} < 1 → estable  |  π₀ = 1-ρ = {1-rho:.4f} ✓")

Matriz Q (M/M/1, primeras 5 filas):
[[-3.  3.  0.  0.  0.]
 [ 5. -8.  3.  0.  0.]
 [ 0.  5. -8.  3.  0.]
 [ 0.  0.  5. -8.  3.]
 [ 0.  0.  0.  5. -8.]]

Forma implícita: q_{n,n+1}=3.0, q_{n,n-1}=5.0  (tasa diagonal: q_00=-3.0, q_11=-8.0)

jmarkov — ¿Es ergódica? True
Distribución estacionaria π∞ (estados 0..5): [0.4015 0.2409 0.1445 0.0867 0.052  0.0312]
Verificación: ρ = λ/μ = 0.60 < 1 → estable  |  π₀ = 1-ρ = 0.4000 ✓


## Escenario 2 — Cola M/M/1/K: clínica dental
<div class='defn'>
Una pequeña clínica dental tiene <strong>1 dentista</strong> y una sala de espera con <strong>3 sillas</strong> (capacidad total K = 4, incluyendo el paciente en atención). Los pacientes llegan a tasa λ = 2 pac/hora. El dentista atiende a tasa μ = 3 pac/hora. Si el sistema está lleno (K pacientes), <strong>el paciente se va y no regresa</strong>.
</div>

**Diferencia con M/M/1:** el espacio de estados es **finito** S = {0, 1, 2, 3, 4}.

Las llegadas se **bloquean** cuando el sistema está lleno:
$$q_{n,n+1} = \lambda \cdot \mathbb{1}(n < K), \quad q_{n,n-1} = \mu \cdot \mathbb{1}(n \geq 1)$$

Con K = 4, λ = 2, μ = 3:
$$Q = \begin{pmatrix} -2 & 2 & 0 & 0 & 0 \\ 3 & -5 & 2 & 0 & 0 \\ 0 & 3 & -5 & 2 & 0 \\ 0 & 0 & 3 & -5 & 2 \\ 0 & 0 & 0 & 3 & -3 \end{pmatrix}$$

In [2]:
lam2, mu2, K = 2.0, 3.0, 4

Q2 = np.zeros((K+1, K+1))
for n in range(K+1):
    if n < K:
        Q2[n, n+1] = lam2
    if n > 0:
        Q2[n, n-1] = mu2
np.fill_diagonal(Q2, -Q2.sum(axis=1))

print("Matriz Q (M/M/1/K=4):")
print(Q2)

mc2 = ctmc(Q2, np.arange(K+1))
pi2 = mc2.steady_state()
print(f"\njmarkov — distribución estacionaria: {pi2.round(4)}")

p_lleno = pi2[K]                          # P(sistema lleno = paciente rechazado)
lam_eff = lam2 * (1 - p_lleno)           # tasa efectiva de ingreso
E_N = sum(n * pi2[n] for n in range(K+1))
print(f"\nP(sistema lleno, paciente rechazado) = {p_lleno:.4f} = {p_lleno*100:.1f}%")
print(f"Tasa efectiva de admisión = λ(1-P_K)  = {lam_eff:.4f} pac/h")
print(f"Número promedio en sistema E[N]        = {E_N:.4f}")
print(f"\nComparación M/M/1 sin límite: ρ = {lam2/mu2:.2f} → estable, E[N] = {(lam2/mu2)/(1-lam2/mu2):.4f}")

Matriz Q (M/M/1/K=4):
[[-2.  2.  0.  0.  0.]
 [ 3. -5.  2.  0.  0.]
 [ 0.  3. -5.  2.  0.]
 [ 0.  0.  3. -5.  2.]
 [ 0.  0.  0.  3. -3.]]

jmarkov — distribución estacionaria: [0.3839 0.2559 0.1706 0.1137 0.0758]

P(sistema lleno, paciente rechazado) = 0.0758 = 7.6%
Tasa efectiva de admisión = λ(1-P_K)  = 1.8483 pac/h
Número promedio en sistema E[N]        = 1.2417

Comparación M/M/1 sin límite: ρ = 0.67 → estable, E[N] = 2.0000


## Escenario 3 — Cola M/M/c: call center
<div class='defn'>
Un centro de llamadas tiene <strong>c = 3 operadores</strong> idénticos. Las llamadas llegan a tasa λ = 10 llamadas/min. Cada operador resuelve una llamada a tasa μ = 4 llamadas/min. Si todos están ocupados, las llamadas esperan en cola sin límite.
</div>

**Novedad:** la tasa de servicio total **depende del estado**:
- Si n ≤ c: hay n operadores activos → tasa de servicio = nμ
- Si n > c: los c operadores están activos → tasa de servicio = cμ

$$\lambda_n = \lambda \quad \forall n, \qquad \mu_n = \min(n, c) \cdot \mu$$

**Condición de estabilidad:** ρ = λ/(cμ) < 1 → ρ = 10/(3×4) = **0.833 < 1** ✓

In [3]:
lam3, mu3, c = 10.0, 4.0, 3
N3 = 20   # truncar

Q3 = np.zeros((N3+1, N3+1))
for n in range(N3+1):
    if n < N3:
        Q3[n, n+1] = lam3
    if n > 0:
        Q3[n, n-1] = min(n, c) * mu3
np.fill_diagonal(Q3, -Q3.sum(axis=1))

print("Primeras filas de Q (M/M/3): la tasa de muerte satura en c·μ =", c*mu3)
print(Q3[:5, :5])

mc3 = ctmc(Q3, np.arange(N3+1))
pi3 = mc3.steady_state()
rho3 = lam3 / (c * mu3)

E_N3  = sum(n * pi3[n] for n in range(N3+1))
E_Nq3 = sum(max(n-c, 0) * pi3[n] for n in range(N3+1))
W3    = E_N3 / lam3
Wq3   = E_Nq3 / lam3
p_esp = sum(pi3[n] for n in range(c, N3+1))   # P(esperar, todos ocupados)

print(f"\njmarkov — M/M/c (c={c}, λ={lam3}, μ={mu3})")
print(f"  ρ efectiva   = {rho3:.4f}")
print(f"  E[N]         = {E_N3:.3f}  llamadas en sistema")
print(f"  E[Nq]        = {E_Nq3:.3f}  llamadas en cola")
print(f"  W            = {W3:.3f}  min en sistema")
print(f"  Wq           = {Wq3:.3f}  min en cola")
print(f"  P(esperar)   = {p_esp:.4f} = {p_esp*100:.1f}%")

Primeras filas de Q (M/M/3): la tasa de muerte satura en c·μ = 12.0
[[-10.  10.   0.   0.   0.]
 [  4. -14.  10.   0.   0.]
 [  0.   8. -18.  10.   0.]
 [  0.   0.  12. -22.  10.]
 [  0.   0.   0.  12. -22.]]

jmarkov — M/M/c (c=3, λ=10.0, μ=4.0)
  ρ efectiva   = 0.8333
  E[N]         = 5.470  llamadas en sistema
  E[Nq]        = 2.983  llamadas en cola
  W            = 0.547  min en sistema
  Wq           = 0.298  min en cola
  P(esperar)   = 0.6942 = 69.4%


## Conclusiones
- La **metodología de 5 pasos** convierte cualquier descripción textual en una CMTC sistemáticamente.
- El **modelo implícito** (tasas como función de los sub-índices) es compacto y programable; el **modelo explícito** (matriz Q completa) es útil para verificación visual.
- La **progresión de complejidad**: M/M/1 → M/M/1/K → M/M/c → dos variables con acoplamiento.
- Con **`jmarkov`**: una vez construida Q, `ctmc(Q, estados)` resuelve automáticamente estado estacionario, análisis transitorio y tiempos de primer pasaje.

| Escenario | Sistema | Variables | |S| | Tasas | Acoplamiento |
|-----------|---------|-----------|-----|-------|--------|
| 1 | M/M/1 | 1 | ∞ | Ctes | — |
| 2 | M/M/1/K | 1 | K+1 | Ctes + borde | — |
| 3 | M/M/c | 1 | ∞ | Dep. estado | — |
| 4 | Reparación | 1 (2→1) | M+1 | Dep. estado | — |
| 5 | Red colas | 2 | (cA+1)(cB+1) | Dep. estado | Sí (bloqueo) |

**Próximo tema:** Análisis transitorio — ¿cómo evoluciona π(t) antes del estado estable?

## Escenario 4 — Reparación de máquinas (dos variables)
<div class='defn'>
Una fábrica tiene <strong>M = 2 máquinas</strong> idénticas y <strong>R = 1 técnico</strong> de reparación. Cada máquina falla con tasa α = 0.5 fallas/hora. El técnico repara a tasa β = 2 reparaciones/hora. Si ambas están dañadas, el técnico repara <strong>una a la vez</strong>.
</div>

**Dos variables:** X₁(t), X₂(t) ∈ {0=ok, 1=dañada} → estado conjunto ∈ {(0,0),(0,1),(1,0),(1,1)}

**Reducción a una variable** (máquinas idénticas):
F(t) = X₁(t) + X₂(t) = número de máquinas dañadas, S = {0, 1, 2}

$$\lambda_f = (M-f)\alpha, \quad \mu_f = \min(f, R)\beta$$

$$Q_{\text{reducida}} = \begin{pmatrix} -2\alpha & 2\alpha & 0 \\ \beta & -(\alpha+\beta) & \alpha \\ 0 & \beta & -\beta \end{pmatrix} = \begin{pmatrix} -1 & 1 & 0 \\ 2 & -2.5 & 0.5 \\ 0 & 2 & -2 \end{pmatrix}$$

In [4]:
# Modelo general: M máquinas, R técnicos
M, R, alpha, beta = 2, 1, 0.5, 2.0

# Construcción implícita de Q (forma reducida)
Q4 = np.zeros((M+1, M+1))
for f in range(M+1):
    lam_f = (M - f) * alpha          # tasa de fallo: máquinas operativas
    mu_f  = min(f, R) * beta         # tasa de reparación: técnicos activos
    if f < M:
        Q4[f, f+1] = lam_f
    if f > 0:
        Q4[f, f-1] = mu_f
np.fill_diagonal(Q4, -Q4.sum(axis=1))

print(f"Modelo de reparación: M={M} máquinas, R={R} técnico, α={alpha}, β={beta}")
print(f"\nMatriz Q (f = # máquinas dañadas):")
for i in range(M+1):
    print(f"  f={i}: {Q4[i].round(2)}")

mc4 = ctmc(Q4, np.array(['0 dañadas', '1 dañada', '2 dañadas']))
pi4 = mc4.steady_state()

print(f"\njmarkov — distribución estacionaria:")
for i, lbl in enumerate(['0 dañadas', '1 dañada', '2 dañadas']):
    print(f"  π∞({lbl}) = {pi4[i]:.4f}  ({pi4[i]*100:.1f}%)")

disp = pi4[0] + pi4[1]
print(f"\nDisponibilidad del sistema (≥1 máquina operativa): {disp*100:.1f}%")
print(f"P(ambas dañadas, producción detenida):             {pi4[2]*100:.1f}%")
E_op = sum((M-f)*pi4[f] for f in range(M+1))
print(f"E[máquinas operativas] = {E_op:.3f}")

# Sensibilidad: ¿qué pasa si agregamos un segundo técnico?
M2, R2 = 2, 2
Q4b = np.zeros((M2+1, M2+1))
for f in range(M2+1):
    lf = (M2-f)*alpha; mf = min(f,R2)*beta
    if f < M2: Q4b[f, f+1] = lf
    if f > 0:  Q4b[f, f-1] = mf
np.fill_diagonal(Q4b, -Q4b.sum(axis=1))
pi4b = ctmc(Q4b, np.arange(M2+1)).steady_state()
print(f"\nCon R=2 técnicos: P(ambas dañadas) = {pi4b[2]*100:.1f}%  (antes: {pi4[2]*100:.1f}%)")

Modelo de reparación: M=2 máquinas, R=1 técnico, α=0.5, β=2.0

Matriz Q (f = # máquinas dañadas):
  f=0: [-1.  1.  0.]
  f=1: [ 2.  -2.5  0.5]
  f=2: [ 0.  2. -2.]

jmarkov — distribución estacionaria:
  π∞(0 dañadas) = 0.6154  (61.5%)
  π∞(1 dañada) = 0.3077  (30.8%)
  π∞(2 dañadas) = 0.0769  (7.7%)

Disponibilidad del sistema (≥1 máquina operativa): 92.3%
P(ambas dañadas, producción detenida):             7.7%
E[máquinas operativas] = 1.538

Con R=2 técnicos: P(ambas dañadas) = 4.0%  (antes: 7.7%)


## Escenario 5 — Red de colas: taller de bicicletas
<div class='defn'>
Un taller tiene <strong>dos estaciones en serie</strong>: Estación A (diagnóstico, μ_A=4/h) y Estación B (reparación, μ_B=3/h). Las bicicletas llegan a tasa λ=2/h. Cada estación tiene espacio para <strong>máximo 2 bicicletas</strong>. Si la estación B está llena, la estación A <strong>queda bloqueada</strong> aunque tenga bicicletas.
</div>

**Variables:** N_A(t) ∈ {0,1,2}, N_B(t) ∈ {0,1,2} → |S| = 9 estados

**Tres tipos de eventos desde estado (a,b):**

| Evento | Destino | Tasa | Condición |
|--------|---------|------|-----------|
| Llegada externa | (a+1, b) | λ | a < 2 |
| Servicio A (pasa a B) | (a−1, b+1) | μ_A | a ≥ 1 y b < 2 |
| Servicio B (sale) | (a, b−1) | μ_B | b ≥ 1 |

<div class='alerta'>
<strong>Bloqueo:</strong> si b = 2 (estación B llena), la estación A no puede despachar aunque tenga bicicletas. Esto <strong>acopla</strong> ambas variables.
</div>

In [5]:
lam5, muA, muB = 2.0, 4.0, 3.0
capA, capB = 2, 2

# Espacio de estados: producto cartesiano {0..capA} x {0..capB}
states5 = [(a, b) for a in range(capA+1) for b in range(capB+1)]
idx = {s: i for i, s in enumerate(states5)}
n = len(states5)

# Construcción implícita de Q (forma de tabla de eventos)
Q5 = np.zeros((n, n))
events5 = [
    ((+1,  0), lam5, lambda a,b: a < capA),           # llegada
    ((-1, +1), muA,  lambda a,b: a >= 1 and b < capB), # servicio A → B
    (( 0, -1), muB,  lambda a,b: b >= 1),              # servicio B → salida
]
for (a, b) in states5:
    for (da, db), rate, cond in events5:
        if cond(a, b):
            dest = (a+da, b+db)
            Q5[idx[(a,b)], idx[dest]] = rate
np.fill_diagonal(Q5, -Q5.sum(axis=1))

print(f"Escenario 5 — Red de colas: {n} estados")
print(f"Estados: {states5}")
print(f"\nMatriz Q (9×9):")
print(np.round(Q5, 1))

mc5 = ctmc(Q5, np.array([str(s) for s in states5]))
pi5 = mc5.steady_state()

print(f"\njmarkov — distribución estacionaria:")
for i, s in enumerate(states5):
    if pi5[i] > 0.005:
        print(f"  π∞{s} = {pi5[i]:.4f}  ({pi5[i]*100:.1f}%)")

# Métricas de rendimiento
p_bloq = sum(pi5[idx[(a,b)]] for (a,b) in states5 if a == capA)
E_NA   = sum(a * pi5[idx[(a,b)]] for (a,b) in states5)
E_NB   = sum(b * pi5[idx[(a,b)]] for (a,b) in states5)
p_Abloq = sum(pi5[idx[(a,b)]] for (a,b) in states5 if a >= 1 and b == capB)  # A bloqueada
print(f"\nMétricas de rendimiento:")
print(f"  P(llegada rechazada, A llena) = {p_bloq:.4f} = {p_bloq*100:.1f}%")
print(f"  P(A bloqueada por B llena)    = {p_Abloq:.4f} = {p_Abloq*100:.1f}%")
print(f"  E[bicicletas en estación A]   = {E_NA:.3f}")
print(f"  E[bicicletas en estación B]   = {E_NB:.3f}")

Escenario 5 — Red de colas: 9 estados
Estados: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]

Matriz Q (9×9):
[[-2.  0.  0.  2.  0.  0.  0.  0.  0.]
 [ 3. -5.  0.  0.  2.  0.  0.  0.  0.]
 [ 0.  3. -5.  0.  0.  2.  0.  0.  0.]
 [ 0.  4.  0. -6.  0.  0.  2.  0.  0.]
 [ 0.  0.  4.  3. -9.  0.  0.  2.  0.]
 [ 0.  0.  0.  0.  3. -5.  0.  0.  2.]
 [ 0.  0.  0.  0.  4.  0. -4.  0.  0.]
 [ 0.  0.  0.  0.  0.  4.  3. -7.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  3. -3.]]

jmarkov — distribución estacionaria:
  π∞(0, 0) = 0.2351  (23.5%)
  π∞(0, 1) = 0.1567  (15.7%)
  π∞(0, 2) = 0.0855  (8.5%)
  π∞(1, 0) = 0.1318  (13.2%)
  π∞(1, 1) = 0.1069  (10.7%)
  π∞(1, 2) = 0.0760  (7.6%)
  π∞(2, 0) = 0.1051  (10.5%)
  π∞(2, 1) = 0.0522  (5.2%)
  π∞(2, 2) = 0.0507  (5.1%)

Métricas de rendimiento:
  P(llegada rechazada, A llena) = 0.2080 = 20.8%
  P(A bloqueada por B llena)    = 0.1267 = 12.7%
  E[bicicletas en estación A]   = 0.731
  E[bicicletas en estación B]   = 0.740


## Ejercicio propuesto — Hospital: urgencias + quirófano
<div class='defn'>
Un hospital tiene una <strong>sala de urgencias con 2 camas</strong> y un <strong>quirófano con 1 mesa</strong>. Los pacientes llegan con tasa λ = 1/hora. Un paciente en urgencias es atendido con tasa μ₁ = 0.5/hora. Con probabilidad p = 0.4 requiere cirugía y pasa al quirófano (si hay espacio); con probabilidad 1−p es dado de alta. La cirugía toma tiempo exponencial con tasa μ₂ = 0.3/hora. Si urgencias está llena, los pacientes se derivan.
</div>

**Variables:** U(t) ∈ {0,1,2}, Q(t) ∈ {0,1} → |S| = 6 estados

**Tabla de eventos:**

| Evento | Δ | Tasa | Condición |
|--------|---|------|-----------|
| Llegada | (+1, 0) | λ | u < 2 |
| Alta directa | (−1, 0) | (1−p)·μ₁·min(u,1) | u ≥ 1 |
| Pasa a quirófano | (−1, +1) | p·μ₁·min(u,1) | u ≥ 1, q = 0 |
| Fin cirugía | (0, −1) | μ₂ | q = 1 |

Construya la matriz Q y calcule con jmarkov:
1. ¿Cuál es la utilización del quirófano?
2. ¿Qué fracción del tiempo urgencias está llena (pacientes derivados)?
3. ¿Cuál es el número promedio de pacientes en el hospital?